In [0]:
# Install both PyTorch and DeepXDE together
%pip install torch torchvision torchaudio
%pip install deepxde
%pip install scipy

# CRITICAL: This restarts the Python process so it recognizes the new libraries
dbutils.library.restartPython()

In [0]:
import os
# Set the backend to pytorch BEFORE importing deepxde
os.environ["DDE_BACKEND"] = "pytorch"

import torch
import deepxde as dde

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from tqdm import tqdm
import mlflow

# Verify installation
print(f"DeepXDE backend: {dde.backend.backend_name}")
print(f"PyTorch version: {torch.__version__}")

In [0]:
# Load the raw files into Spark DataFrames
raw_metadata = spark.read.csv("/Volumes/workspace/default/pinn/cleaned_battery_dataset/metadata.csv", header=True, inferSchema=True)

# Register as Temporary Views so we can use SQL
raw_metadata.createOrReplaceTempView("raw_metadata")

# Save as a permanent Bronze Table
raw_metadata.write.format("delta").mode("overwrite").saveAsTable("bronze_battery_metadata")

In [0]:
%sql
SELECT * 
FROM raw_metadata 
LIMIT 50;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_discharge_metadata AS
WITH exploded_time AS (
  SELECT 
    battery_id, 
    type, 
    filename, 
    ambient_temperature, 
    start_time,
    capacity,
    regexp_extract_all(start_time, r'(\d+\.?\d*([eE][+-]?\d+)?)') as parts
  FROM bronze_battery_metadata
  WHERE battery_id NOT IN ('B0049', 'B0050', 'B0051', 'B0052')
),
formatted_time AS (
  SELECT *,
    CAST(CAST(parts[0] AS DOUBLE) AS INT) as y,
    CAST(CAST(parts[1] AS DOUBLE) AS INT) as m,
    CAST(CAST(parts[2] AS DOUBLE) AS INT) as d,
    CAST(CAST(parts[3] AS DOUBLE) AS INT) as hr,
    CAST(CAST(parts[4] AS DOUBLE) AS INT) as mi,
    CAST(parts[5] AS DOUBLE) as sec
  FROM exploded_time
)
SELECT 
  battery_id, type, filename, ambient_temperature, capacity,
  to_timestamp(concat(y, '-', m, '-', d, ' ', hr, ':', mi, ':', sec), 'yyyy-M-d H:m:s.SSS') AS start_datetime,
  ROW_NUMBER() OVER(PARTITION BY battery_id ORDER BY y, m, d, hr, mi, sec) AS cycle_number
FROM formatted_time
WHERE type = 'discharge';

In [0]:
%sql
SELECT *
FROM silver_discharge_metadata 
WHERE start_datetime IS NOT NULL
LIMIT 50;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_impedance_metadata AS
WITH cleaned_strings AS (
  SELECT 
    battery_id,
    Re,
    Rct,
    type,
    -- Strip brackets and split by one or more spaces
    split(trim(regexp_replace(start_time, r'\[|\]', '')), ' +') as parts
  FROM raw_metadata
  WHERE battery_id NOT IN ('B0049', 'B0050', 'B0051', 'B0052')
),
formatted_time AS (
  SELECT 
    *,
    -- Convert indices to numbers, handling potential scientific notation
    CAST(CAST(parts[0] AS DOUBLE) AS INT) as y,
    CAST(CAST(parts[1] AS DOUBLE) AS INT) as m,
    CAST(CAST(parts[2] AS DOUBLE) AS INT) as d,
    CAST(CAST(parts[3] AS DOUBLE) AS INT) as hr,
    CAST(CAST(parts[4] AS DOUBLE) AS INT) as mi,
    CAST(parts[5] AS DOUBLE) as sec
  FROM cleaned_strings
  -- Ensure we actually have 6 parts (Y, M, D, H, M, S)
  WHERE size(parts) >= 6
)
SELECT 
  battery_id, 
  Re,
  Rct,
  to_timestamp(
    concat(y, '-', m, '-', d, ' ', hr, ':', mi, ':', sec), 
    'yyyy-M-d H:m:s.SSS'
  ) AS start_datetime,
  ROW_NUMBER() OVER(PARTITION BY battery_id ORDER BY y, m, d, hr, mi, sec) AS cycle_number
FROM formatted_time
WHERE type = 'impedance';

In [0]:
%sql
SELECT *
FROM silver_impedance_metadata 
WHERE start_datetime IS NOT NULL
LIMIT 50;

In [0]:
%sql
CREATE OR REPLACE TABLE gold_battery_pinn_database AS
SELECT 
  d.battery_id, 
  d.type, 
  d.filename, 
  d.ambient_temperature, 
  d.start_datetime, 
  d.cycle_number,
  CAST(d.capacity AS FLOAT) AS capacity,
  CAST(i.Re AS FLOAT) AS Re,   
  CAST(i.Rct AS FLOAT) AS Rct
FROM silver_discharge_metadata d
LEFT JOIN silver_impedance_metadata i 
  ON d.battery_id = i.battery_id 
  AND i.start_datetime <= d.start_datetime
-- Keep only the first occurrence of each filename
-- Sorting by start_datetime ensures we keep the most logical chronological record
QUALIFY ROW_NUMBER() OVER(PARTITION BY d.filename ORDER BY d.start_datetime ASC) = 1;

In [0]:
%sql
SELECT *
FROM gold_battery_pinn_database 
WHERE start_datetime IS NOT NULL
ORDER BY filename
LIMIT 50;

In [0]:
%sql
SELECT `battery_id`, COUNT(*) AS `count`
FROM `workspace`.`default`.`gold_battery_pinn_database`
WHERE `ambient_temperature` = 24
GROUP BY `battery_id`
ORDER BY `count` DESC;

In [0]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

# --- 1. Configurations ---
cutoff_map = {
    'B0005': 2.7,'B0006': 2.5,'B0007': 2.2,'B0018': 2.5,'B0025': 2.0, 'B0026': 2.2, 
    'B0027': 2.5,'B0028': 2.7,'B0029': 2.0,'B0030': 2.2,'B0031': 2.5,'B0032': 2.7,
    'B0033': 2.0,'B0034': 2.2,'B0036': 2.7,'B0038': 2.2,'B0039': 2.5,'B0040': 2.7,
    'B0041': 2.0,'B0042': 2.2,'B0043': 2.5,'B0044': 2.7,'B0045': 2.0,'B0046': 2.2,
    'B0047': 2.5,'B0048': 2.7,'B0053': 2.0,'B0054': 2.2,'B0055': 2.5,'B0056': 2.7
}

target_batteries = ['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027', 
                    'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034', 
                    'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043', 
                    'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0053', 'B0054', 
                    'B0055', 'B0056']

# --- 2. Query the Gold Database ---
# Instead of pd.read_csv + pd.merge_asof, we fetch the pre-joined data from SQL
query = f"""
    SELECT * FROM gold_battery_pinn_database 
    WHERE battery_id IN ({','.join([f"'{b}'" for b in target_batteries])})
"""
discharge_metadata = spark.sql(query).toPandas()

# Run once outside the loop
max_caps_df = spark.sql("""
    SELECT battery_id, max(capacity) as max_cap 
    FROM `workspace`.`default`.`gold_battery_pinn_database` 
    GROUP BY battery_id
""").toPandas().set_index('battery_id')

processed_dfs = []

# --- 3. Process Discharge Cycles ---
for _, row in tqdm(discharge_metadata.iterrows(), total=len(discharge_metadata)):
    file_path = f"/Volumes/workspace/default/pinn/cleaned_battery_dataset/data/{row['filename']}"
    if not os.path.exists(file_path): 
        continue

    # Load raw time-series data
    df = pd.read_csv(file_path).copy()
    
    # Truncate at voltage limit
    v_limit = cutoff_map.get(row['battery_id'], 2.7)
    cutoff_idx = df[df['Voltage_measured'] < v_limit].index.min()
    truncated_df = df if pd.isna(cutoff_idx) else df.iloc[:cutoff_idx].copy()

    if len(truncated_df) < 50: 
        continue
    
    # Calculate Capacity & SoC
    db_capacity = float(row['capacity'])
    max_capacity = max_caps_df.loc[row['battery_id'], 'max_cap']

    # Only process cycles with valid capacity
    if db_capacity > 1.4:
        truncated_df['Time_diff_hr'] = truncated_df['Time'].diff().fillna(0) / 3600
        truncated_df['Delta_Q'] = abs(truncated_df['Current_measured'] * truncated_df['Time_diff_hr'])
        truncated_df['Cumulative_Q'] = truncated_df['Delta_Q'].cumsum()
        truncated_df['SoC'] = 100 * (1 - truncated_df['Cumulative_Q'] / db_capacity)
        
        # Downsample to 50 points
        num_bins = 50
        truncated_df['bin'] = pd.cut(truncated_df.index, bins=num_bins, labels=False)
        
        agg_cycle = truncated_df.groupby('bin').agg({
            'Time': 'mean',
            'Voltage_measured': 'mean',
            'Current_measured': 'mean',
            'Temperature_measured': 'mean',
            'SoC': 'mean'
        }).reset_index()

        # Add Database-derived Metadata
        agg_cycle['battery_id'] = row['battery_id']
        agg_cycle['cycle_number'] = row['cycle_number']
        agg_cycle['SoH'] = float((db_capacity / max_capacity) * 100)
        agg_cycle['Re'] = float(row['Re']) if pd.notna(row['Re']) else None
        agg_cycle['Rct'] = float(row['Rct']) if pd.notna(row['Rct']) else None
        agg_cycle['Time_normalized'] = agg_cycle['Time'] / agg_cycle['Time'].max()

        processed_dfs.append(agg_cycle)

# --- 4. Final Dataset Creation ---
if processed_dfs:
    full_dataset = pd.concat(processed_dfs, ignore_index=True)
    
    # Save the processed data back into a NEW Gold Table for the PINN
    spark_processed = spark.createDataFrame(full_dataset)
    spark_processed.write.format("delta").mode("overwrite").saveAsTable("final_pinn_training_data")
    
    print("Training database 'final_pinn_training_data' created with target batteries.")

In [0]:
spark.sql("""
    SELECT * FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cycle_number ORDER BY SoH DESC) as rn
        FROM final_pinn_training_data
        //WHERE battery_id = 'B0034'
    )
    WHERE rn = 1
    ORDER BY SoH DESC
    LIMIT 10
""").show()

In [0]:
# This code loads battery metadata, filters and processes discharge cycles, merges impedance data, and generates a cleaned, downsampled dataset for PINN training.

# Load and preprocess metadata
metadata = pd.read_csv("/Volumes/workspace/default/pinn/cleaned_battery_dataset/metadata.csv")
metadata['battery_id'] = metadata['battery_id'].astype(str)

# Exclude problematic batteries
excluded_batteries = ['B0049', 'B0050', 'B0051', 'B0052']

cutoff_map = {
    'B0005': 2.7,'B0006': 2.5,'B0007': 2.2,'B0018': 2.5,'B0025': 2.0, 'B0026': 2.2, 
    'B0027': 2.5,'B0028': 2.7,'B0029': 2.0,'B0030': 2.2,'B0031': 2.5,'B0032': 2.7,
    'B0033': 2.0,'B0034': 2.2,'B0036': 2.7,'B0038': 2.2,'B0039': 2.5,'B0040': 2.7,
    'B0041': 2.0,'B0042': 2.2,'B0043': 2.5,'B0044': 2.7,'B0045': 2.0,'B0046': 2.2,
    'B0047': 2.5,'B0048': 2.7,'B0053': 2.0,'B0054': 2.2,'B0055': 2.5,'B0056': 2.7
}

target_batteries = ['B0005', 'B0006', 'B0007', 'B0018', 'B0025', 'B0026', 'B0027', 
                    'B0028', 'B0029', 'B0030', 'B0031', 'B0032', 'B0033', 'B0034', 
                    'B0036', 'B0038', 'B0039', 'B0040', 'B0041', 'B0042', 'B0043', 
                    'B0044', 'B0045', 'B0046', 'B0047', 'B0048', 'B0053', 'B0054', 
                    'B0055', 'B0056']

# Filter to only discharge cycles, excluding the listed batteries
discharge_metadata = metadata[
    (metadata['type'] == 'discharge') & 
    (~metadata['battery_id'].isin(excluded_batteries))
].copy()

# Assign a sequential cycle number per battery
discharge_metadata['cycle_number'] = discharge_metadata.groupby('battery_id').cumcount() + 1

processed_dfs = []

# NASA start_time is a string list: "[2010. 7. 21. ...]". We must parse it.
def parse_nasa_time(time_str):
    try:
        clean = time_str.replace('[', '').replace(']', '').strip().split()
        parts = [float(p) for p in clean]
        # Format: Year, Month, Day, Hour, Minute, Second
        return pd.to_datetime(f"{int(parts[0])}-{int(parts[1])}-{int(parts[2])} {int(parts[3])}:{int(parts[4])}:{parts[5]}", errors='coerce')
    except:
        return pd.NaT

metadata['datetime'] = metadata['start_time'].apply(parse_nasa_time)
metadata = metadata.sort_values(['battery_id', 'datetime'])

# Separate impedance and discharge
imp_meta = metadata[metadata['type'] == 'impedance'][['datetime', 'battery_id', 'Re', 'Rct']].dropna(subset=['Re'])
dis_meta = metadata[metadata['type'] == 'discharge'].drop(columns=['Re', 'Rct']).copy()

# Perform the merge: find the nearest PREVIOUS impedance test for each discharge
discharge_metadata = pd.merge_asof(
    dis_meta.sort_values('datetime'),
    imp_meta.sort_values('datetime'),
    on='datetime',
    by='battery_id',
    direction='backward'
)

# Handle the deprecated fillna method
discharge_metadata = discharge_metadata.ffill().bfill() 

# Filter excluded batteries BEFORE calculating cycles
discharge_metadata = discharge_metadata[~discharge_metadata['battery_id'].isin(excluded_batteries)].copy()

# NOW assign the sequential cycle number
# This ensures 'cycle_number' exists during the loop
discharge_metadata['cycle_number'] = discharge_metadata.groupby('battery_id').cumcount() + 1

# Process each discharge cycle
for _, row in tqdm(discharge_metadata.iterrows(), total=len(discharge_metadata)):
    file_path = f"/Volumes/workspace/default/pinn//cleaned_battery_dataset/data/{row['filename']}"
    if not os.path.exists(file_path): continue

    df = pd.read_csv(file_path).copy()
    
    # Truncate data at the first point where voltage < v_limit or 2.7
    v_limit = cutoff_map.get(row['battery_id'], 2.7)
    cutoff_idx = df[df['Voltage_measured'] < v_limit].index.min()
    truncated_df = df if pd.isna(cutoff_idx) else df.iloc[:cutoff_idx].copy()

    # Basic data health check
    if len(truncated_df) < 50: continue
    
    # DOWNSAMPLING: Keeping Time (The Independent Variable)
    # Calculate Capacity & SoC
    truncated_df['Time_diff_hr'] = truncated_df['Time'].diff().fillna(0) / 3600
    truncated_df['Delta_Q'] = truncated_df['Current_measured'] * truncated_df['Time_diff_hr']
    capacity = abs(truncated_df['Delta_Q'].sum())
    
    if capacity > 1.4:
        # Calculate SoC (from 100% to 0% for this specific cycle)
        truncated_df['Cumulative_Q'] = abs(truncated_df['Delta_Q'].cumsum())
        truncated_df['SoC'] = 100 * (1 - truncated_df['Cumulative_Q'] / capacity)
        
        # DOWNSAMPLING: 50 bins
        num_bins = 50
        truncated_df['bin'] = pd.cut(truncated_df.index, bins=num_bins, labels=False)
        
        # IMPORTANT: SoC must be in this .agg() dictionary
        agg_cycle = truncated_df.groupby('bin').agg({
            'Time': 'mean',
            'Voltage_measured': 'mean',
            'Current_measured': 'mean',
            'Temperature_measured': 'mean',
            'SoC': 'mean' 
        }).reset_index()

        # Add Metadata
        agg_cycle['battery_id'] = row['battery_id']
        agg_cycle['cycle_number'] = row['cycle_number']
        agg_cycle['SoH'] = (capacity / 2.0) * 100
        agg_cycle['Re'] = row['Re']
        agg_cycle['Rct'] = row['Rct']
        agg_cycle['Time_normalized'] = agg_cycle['Time'] / agg_cycle['Time'].max()

        processed_dfs.append(agg_cycle)

if processed_dfs:
    full_dataset = pd.concat(processed_dfs)
    full_dataset.to_csv("battery_health_dataset.csv", index=False)

In [0]:
# ---------------------------------------------------------
# 1. Physics Constants & Parameters
# ---------------------------------------------------------
R_gas = 8.314        # Gas constant (J/mol·K)
Ea_sei = 1.35e5      # Activation energy (J/mol)
H_sei = 2.57e5       # Enthalpy of reaction (J/kg)
rho = 2000.0         # Density (kg/m3)
Cp = 1000.0          # Heat capacity (J/kg·K)
k_cond = 1.0         # Thermal conductivity (W/m·K)
T_ref = 298.15       # Baseline Kelvin (25°C)

# Define A_sei as a learnable variable (starting at 10^15)
log_A_sei = dde.Variable(15.0)

In [0]:
# ---------------------------------------------------------
#  Data Preparation & Database Querying
# ---------------------------------------------------------
# Load data from the processed SQL database 
query = "SELECT * FROM final_pinn_training_data"
df = spark.sql(query).toPandas()

# Constants for scaling
t_max_seconds = df['Time'].max()

# Strictly monotonic interpolation:
# Since we are training a global model across many batteries, 
# we interpolate the average characteristic profiles for Current, Voltage, and SoC.
# This provides the "Representative Discharge" the PINN learns from.

# Sort by normalized time to ensure 1D interpolation works correctly
df_sorted = df.sort_values('Time_normalized')

# Group by normalized time to handle duplicates at specific time steps
df_agg = df_sorted.groupby('Time_normalized').agg({
    'Current_measured': 'mean',
    'Voltage_measured': 'mean',
    'SoC': 'mean'
}).reset_index()

t_pts = df_agg['Time_normalized'].values
interp_I = interp1d(t_pts, df_agg['Current_measured'], fill_value="extrapolate")
interp_V = interp1d(t_pts, df_agg['Voltage_measured'], fill_value="extrapolate")
interp_SoC = interp1d(t_pts, df_agg['SoC'], fill_value="extrapolate")

def get_real_time_physics(t_norm):
    """
    Mapping function for the PINN's physics loss.
    Converts normalized time [0,1] into the specific I, V, and OCV 
    required to solve the heat equation during training.
    """
    t_np = t_norm.detach().cpu().numpy().flatten()
    
    # Map normalized time to physics values
    I_val = torch.tensor(interp_I(t_np), dtype=torch.float32).view(-1, 1).to(t_norm.device)
    V_val = torch.tensor(interp_V(t_np), dtype=torch.float32).view(-1, 1).to(t_norm.device)
    SoC_val = torch.tensor(interp_SoC(t_np), dtype=torch.float32).view(-1, 1).to(t_norm.device)
    
    # NASA Battery OCV Approximation: Linear mapping based on SoC
    # U_ocv = V_ocv_intercept + V_ocv_slope * SoC
    U_ocv = 2.7 + 0.015 * SoC_val 
    
    return I_val, V_val, U_ocv

In [0]:
# ---------------------------------------------------------
# 3.4D PDE System
# ---------------------------------------------------------

def sei_thermal_pde(x, y):
    T_delta = y[:, 0:1] 
    c = y[:, 1:2]
    
    # Reconstruct T and clamp to prevent the 'exp' from exploding
    T = T_delta * 50.0 + T_ref 
    # Hard clamp between 0°C and 400°C for the math stability
    T_safe = torch.clamp(T, min=270.0, max=600.0)
    
    # Guard concentration: SEI health cannot be negative
    c_safe = torch.clamp(c, min=0.0, max=1.1)

    dT_t = dde.grad.jacobian(y, x, i=0, j=3) / t_max_seconds
    dc_t = dde.grad.jacobian(y, x, i=1, j=3) / t_max_seconds
    dT_xx = dde.grad.hessian(y, x, i=0, j=0)

    I, V, U_ocv = get_real_time_physics(x[:, 3:4])
    actual_A_sei = torch.pow(10.0, log_A_sei)
    joule_heat = torch.abs(I * (V - U_ocv)) 

    # The denominator guard (1e-6) prevents division by zero
    # The rate clamp (1e5) prevents 'Inf' from reaching the optimizer
    exponent = -Ea_sei / (R_gas * T_safe + 1e-6)
    rate = actual_A_sei * torch.clamp(c, min=0, max=1) * torch.exp(torch.clamp(exponent, min=-50, max=50))
    # Multiply the heat generation term by 2 or 5 to see the effect
    res_thermal = (dT_t * 50.0) - (k_cond * dT_xx * 50.0 + joule_heat + H_sei*rate) / (rho * Cp)
    res_sei = dc_t + rate

    return [res_thermal, res_sei]

In [0]:
# ---------------------------------------------------------
# 4. Geometry & Robust Initial Conditions
# ---------------------------------------------------------

# 1. Define Geometry
param_geom = dde.geometry.Hypercube(xmin=[0, 0.5, 0.0], xmax=[1, 1.2, 12.0])
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(param_geom, timedomain)

# 2. Fuzzy ICs
def on_initial_condition(x, on_initial):
    return on_initial and np.isclose(x[3], 0, atol=1e-5)

ic_T = dde.icbc.IC(geomtime, lambda x: 0, on_initial_condition, component=0)
ic_c = dde.icbc.IC(geomtime, lambda x: 1.0, on_initial_condition, component=1)

# --- ADD THIS BLOCK TO DEFINE THE VARIABLES ---
# Ensure you are using the cleaned dataframe from your previous step
obs_points = np.column_stack((
    np.ones(len(df)),             # Space (x=1)
    df['SoH'].values / 100.0,     # SoH
    df['Re'].values * 10.0,       # Re
    df['Time_normalized'].values  # Time (t)
))

obs_values = ((df['Temperature_measured'].values - T_ref) / 50.0).reshape(-1, 1)
# ----------------------------------------------

# 3. Synchronize Data Types
obs_points = obs_points.astype(np.float32)
obs_values = obs_values.astype(np.float32)

observe_T = dde.icbc.PointSetBC(obs_points, obs_values, component=0)

In [0]:
import mlflow
import torch
import deepxde as dde

# ---------------------------------------------------------
# 5. Model Architecture & Training
# ---------------------------------------------------------
# A. Updated Metrics for Battery Digital Twin
def mae_temp_metric(y_true, y_pred):
    # If DeepXDE is testing on PDE points (no NASA labels), y_true is None
    if y_true is None: 
        return torch.tensor(0.0)
    
    # y_pred has 2 columns: [T_delta, c]
    # y_true has 1 column: [Temperature_measured]
    # We slice y_pred to only compare the Temperature component
    return torch.mean(torch.abs(y_true - y_pred[:, 0:1]))

def l2_temp_error(y_true, y_pred):
    if y_true is None: 
        return torch.tensor(0.0)
    
    # Compare only the temperature components
    return torch.norm(y_true - y_pred[:, 0:1]) / (torch.norm(y_true) + 1e-7)

# B. Data Object Configuration
# num_test=500 provides the 'y_true' values needed for the metrics to function
data = dde.data.TimePDE(
    geomtime, 
    sei_thermal_pde, 
    [ic_T, ic_c, observe_T], 
    num_domain=5000,      # Increase domain points
    num_boundary=500,     # Explicitly sample the boundaries
    anchors=obs_points,
    num_test=2000
)

# C. Network Architecture
# 4 Inputs: [Space, SoH, Re, Time] -> 4 Hidden Layers -> 2 Outputs: [T_delta, c]
# Use the float32 casting for the network weights
net = dde.nn.FNN([4] + [64] * 4 + [2], "tanh", "Glorot normal")
net.float() # Force weights to float32
model = dde.model.Model(data, net)

# D. Custom MLflow Logging Callback
class MLflowLoggingCallback(dde.callbacks.Callback):
    def on_epoch_end(self):
        # Log every 1000 steps to avoid cluttering Databricks logs
        if self.model.train_state.step % 1000 == 0:
            # Log the learned physical parameter A_sei
            current_A = 10**log_A_sei.detach().item()
            mlflow.log_metric("A_sei_value", current_A, step=self.model.train_state.step)
            
            # Log all 5 loss components (PDE 1, PDE 2, IC 1, IC 2, Data)
            for i, loss in enumerate(self.model.train_state.loss_train):
                mlflow.log_metric(f"loss_comp_{i}", float(loss), step=self.model.train_state.step)
            
            # Log the metrics (MAE and L2)
            if self.model.train_state.metrics_test is not None:
                mlflow.log_metric("MAE_Kelvin_Scaled", float(self.model.train_state.metrics_test[0]), step=self.model.train_state.step)
                mlflow.log_metric("L2_Relative_Error", float(self.model.train_state.metrics_test[1]), step=self.model.train_state.step)

# E. Execution within MLflow Context
with mlflow.start_run(run_name="Global_Battery_Digital_Twin"):

    
    print("Starting Adam Optimization...")
    # Add this to Stage 1 model.compile or model.train
    # It reduces the LR by half every 5,000 iterations
    # 1. Stage 1: Adam Optimizer
    model.compile(
        "adam", 
        lr=1e-3, # Increased slightly to help it unfreeze
        loss_weights=[1, 1, 1, 1, 100], # Force focus on NASA data
        external_trainable_variables=log_A_sei,
        metrics=[mae_temp_metric, l2_temp_error] # Use the new names here
    )

    # 2. Stage 2: L-BFGS Fine-Tuning
    print("Starting L-BFGS Refinement...")
    model.compile("L-BFGS", external_trainable_variables=log_A_sei)
    dde.optimizers.config.set_LBFGS_options(maxiter=1000) 
    model.train()
    
    # Increase the tolerance so L-BFGS doesn't give up on the first 'nan'
    dde.optimizers.config.set_LBFGS_options(maxiter=1000, ftol=1e-9)
    losshistory, train_state = model.train()

    # 3. Final Artifact Logging
    final_A = 10**log_A_sei.detach().item()
    mlflow.log_param("Final_Calibrated_A_sei", f"{final_A:.2e}")
    
    # Save model weights to DBFS/MLflow
    torch.save(net.state_dict(), "battery_pinn_model.pt")
    mlflow.log_artifact("battery_pinn_model.pt")

print(f"Training successfully completed. Frequency factor calibrated to: {final_A:.2e}")
# 1. Save the model weights to the Databricks File Store (DBFS)
model.save("global_pinn_model") 



In [0]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import mlflow
from mlflow.tracking import MlflowClient

# 1. SET BACKEND BEFORE IMPORTING DEEPXDE
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde

# Constants for Denormalization
T_ref = 298.15
T_scale = 50.0

# 2. RE-DEFINE ARCHITECTURE (Verified: 4 Hidden Layers)
def dummy_pde(x, y):
    return [dde.grad.jacobian(y, x, i=0, j=1), dde.grad.jacobian(y, x, i=1, j=1)]

geom = dde.geometry.Interval(0, 1)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)
data = dde.data.TimePDE(geomtime, dummy_pde, [], num_domain=100)
net = dde.nn.FNN([4] + [64] * 4 + [2], "tanh", "Glorot normal")
net.float()


model = dde.model.Model(data, net)
data = dde.data.TimePDE(geomtime, dummy_pde, [], num_domain=100)
model = dde.model.Model(data, net)

# 3. DOWNLOAD & RESTORE
client = MlflowClient()
run_id = "867e770b5c7c473cac7e3020038df75d"
artifact_path = "battery_pinn_model.pt"

local_path = client.download_artifacts(run_id, artifact_path)
checkpoint = torch.load(local_path, map_location=torch.device('cpu'))

# Robust load for DeepXDE format vs standard PyTorch
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.net.load_state_dict(checkpoint["model_state_dict"])
else:
    model.net.load_state_dict(checkpoint)


model.compile(
    "adam", 
    lr=1e-3, # Increased slightly to help it unfreeze
    loss_weights=[1, 1, 1, 1, 100], # Force focus on NASA data
    external_trainable_variables=log_A_sei,
    metrics=[mae_temp_metric, l2_temp_error] # Use the new names here
)

print("✅ Model Restored. Architecture: 4 Hidden Layers. Backend: PyTorch.")

# 4. DIGITAL TWIN SIMULATION
test_t = np.linspace(0, 1, 100)
s_soh, s_re = 0.8, 1.5  # Simulate a 80% health battery

# Predict Internal (x=0) vs Surface (x=1)
core_in = np.column_stack((
    np.zeros(100),           # Space (x=0)
    np.full(100, s_soh),     # SoH
    np.full(100, s_re),      # Re
    test_t                   # Time
))

surf_in = np.column_stack((
    np.ones(100),            # Space (x=1)
    np.full(100, s_soh),     # SoH
    np.full(100, s_re),      # Re
    test_t                   # Time
))

# --- RE-RUN PREDICTION WITH CLAMPING ---
res_c = model.predict(core_in)
res_s = model.predict(surf_in)

# Force physical constraints for visualization
t_core_c = (res_c[:, 0] * T_scale + T_ref) - 273.15
t_surf_c = (res_s[:, 0] * T_scale + T_ref) - 273.15
sei_h = res_c[:, 1]

# Force physical constraints (Battery won't be colder than ambient/T_ref)
t_core_c = np.maximum(t_core_c, T_ref - 273.15) 
t_surf_c = np.maximum(t_surf_c, T_ref - 273.15)

# 5. VISUALIZATION
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot Temperatures on Left Y-Axis
ax1.plot(test_t, t_core_c, 'r-', label='Digital Twin: Core (Inferred)', linewidth=2)
ax1.plot(test_t, t_surf_c, 'b--', label='Digital Twin: Surface (Matched)', linewidth=2)
ax1.set_ylabel('Temperature (°C)', fontweight='bold')
ax1.set_xlabel('Discharge Cycle Progress (0=Start, 1=End)', fontweight='bold')
ax1.set_ylim(15, 65) # Adjusted for realistic battery operation
ax1.grid(True, which='both', linestyle='--', alpha=0.5)
ax1.legend(loc='upper left')

# Plot SEI Health on Right Y-Axis
ax2 = ax1.twinx()
ax2.plot(test_t, sei_h, 'g-', label='SEI Stability (c)', linewidth=3)
ax2.set_ylabel('SEI Health (Normalized)', color='g', fontweight='bold')
ax2.set_ylim(0, 1.1)
ax2.legend(loc='upper right')

plt.title(f"Validated Battery Digital Twin: SoH {s_soh*100}%", fontsize=14)
plt.tight_layout()
plt.show()


In [0]:
# --- Simulation of Ignition Threshold ---
# We force the temperature to rise to see when 'c' collapses
oven_t_c = np.linspace(25, 150, 100) # Simulating an oven test 25C to 150C
oven_t_norm = (oven_t_c + 273.15 - T_ref) / 50.0

# Input: [x=0, SoH=0.8, Re=0.5, t_norm]
# Note: We use the oven_t_norm in the temperature input slot to test sensitivity
abuse_test = np.column_stack((
    np.zeros(100),               # Core
    np.full(100, 0.8),           # 80% SoH
    np.full(100, 0.5),           # Standard Re
    np.linspace(0, 1, 100)       # Time steps
))

'''preds = model.predict(abuse_test)
sei_stability = preds[:, 1]
predicted_t = (preds[:, 0] * 50.0 + T_ref) - 273.15'''
# 1. Prediction from the DT

res = model.predict(abuse_test)
core_temp = res[:, 0] * 50 + (T_ref - 273.15)
sei_health = res[:, 1]

# 2. Logic based on your "Point of No Return" Graph
if core_temp.max() > 80.0 or sei_health.min() < 0.985:
    status = "DANGER: THERMAL RUNAWAY PREDICTED"
elif core_temp.max() > 60.0:
    status = "WARNING: ACCELERATED AGING DETECTED"
else:
    status = "SAFE"

print(f"Status: {status}")
# Visualisation
plt.figure(figsize=(10, 5))
plt.plot(core_temp, sei_health, 'g-', label='SEI Health')
plt.axvline(x=80, color='r', linestyle='--', label='Critical Threshold (80°C)')
plt.xlabel('Internal Temperature (°C)')
plt.ylabel('Chemical Stability (c)')
plt.title('Digital Twin: Predicting the Point of No Return')
plt.axhline(y=0.985, color='orange', linestyle=':', label='Chemical Failure Limit')
plt.legend()
plt.show()